# Act 3 — Side-by-Side Evaluation Comparison via EvalHub MCP

**Edit Cell 1 only.** Everything else is driven by the configuration you set there.

This notebook calls the EvalHub MCP server at `localhost:3001` using JSON-RPC 2.0 over HTTP.  
See `mcp_client.py` — the entire MCP client is 60 lines of `requests`.

In [ ]:
# Cell 1 — Configuration
# ─────────────────────────────────────────────────────────────────────────────
# Researchers: edit APPROACH_A and APPROACH_B. Nothing else needs to change.
# ─────────────────────────────────────────────────────────────────────────────
import os

MODEL_URL  = os.environ.get("MODEL_ENDPOINT", "https://openrouter.ai/api/v1")
MODEL_NAME = os.environ.get("MODEL_NAME",     "liquid/lfm-2.5-1.2b-instruct:free")
MCP_URL    = "http://localhost:3001"

APPROACH_A = {
    "name":        "baseline-bbq",
    "benchmark":   "bbq_generate",
    "provider":    "lm_evaluation_harness",
    "description": "BBQ intersectional bias — established baseline (Parrish et al. 2022)",
}

APPROACH_B = {
    "name":        "novel-irr-safety",
    "benchmark":   "icad2026-irr-safety",
    "provider":    "lm_evaluation_harness",
    "description": "IRR-validated safety benchmark — novel contribution (α = 0.81)",
}

print(f"Model:       {MODEL_NAME}")
print(f"Backend URL: {MODEL_URL}")
print(f"MCP server:  {MCP_URL}")

In [ ]:
# Cell 2 — Connect to EvalHub MCP server and verify tools
import sys
sys.path.insert(0, ".")
from mcp_client import EvalHubMCPClient

mcp = EvalHubMCPClient(url=MCP_URL)

try:
    tools = mcp.list_tools()
    print(f"Connected to EvalHub MCP at {MCP_URL}")
    print(f"Available tools: {[t['name'] for t in tools]}")
except Exception as exc:
    print(f"ERROR: {exc}")
    print("Is evalhub-mcp running? → bash 04-mcp-compare/setup.sh")

In [ ]:
# Cell 3 — Submit both approaches

print("Submitting evaluations...")

result_a = mcp.submit_evaluation(
    name=APPROACH_A["name"],
    model_url=MODEL_URL,
    model_name=MODEL_NAME,
    benchmark=APPROACH_A["benchmark"],
    provider=APPROACH_A["provider"],
)
job_id_a = result_a.get("job_id") or result_a.get("id") or str(result_a)
print(f"  Approach A ({APPROACH_A['benchmark']}): job_id = {job_id_a}")

result_b = mcp.submit_evaluation(
    name=APPROACH_B["name"],
    model_url=MODEL_URL,
    model_name=MODEL_NAME,
    benchmark=APPROACH_B["benchmark"],
    provider=APPROACH_B["provider"],
)
job_id_b = result_b.get("job_id") or result_b.get("id") or str(result_b)
print(f"  Approach B ({APPROACH_B['benchmark']}): job_id = {job_id_b}")

In [ ]:
# Cell 4 — Poll until both jobs complete
import time

POLL_INTERVAL = 5
TIMEOUT = 600

job_ids = [job_id_a, job_id_b]
start = time.time()

while time.time() - start < TIMEOUT:
    statuses = {jid: mcp.get_job_status(jid) for jid in job_ids}
    n_done = sum(1 for s in statuses.values() if s.get("status") == "COMPLETED")
    elapsed = int(time.time() - start)
    print(f"  {n_done}/{len(job_ids)} completed  ({elapsed}s elapsed)", end="\r")
    if n_done == len(job_ids):
        print(f"\n  Both jobs completed in {elapsed}s.")
        break
    time.sleep(POLL_INTERVAL)
else:
    raise TimeoutError(f"Jobs did not complete within {TIMEOUT}s.")

In [ ]:
# Cell 5 — Extract scores from job results

def extract_score(status):
    for key in ("overall_score", "score", "result"):
        val = status.get(key)
        if isinstance(val, (int, float)):
            return float(val)
    results = status.get("results") or status.get("benchmarks") or []
    if isinstance(results, list):
        scores = [r.get("score") or r.get("overall_score") for r in results if isinstance(r, dict)]
        scores = [s for s in scores if isinstance(s, (int, float))]
        if scores:
            return sum(scores) / len(scores)
    return None

score_a = extract_score(statuses[job_id_a])
score_b = extract_score(statuses[job_id_b])

print(f"Approach A — {APPROACH_A['description']}")
print(f"  Score: {score_a}")
print()
print(f"Approach B — {APPROACH_B['description']}")
print(f"  Score: {score_b}")

In [ ]:
# Cell 6 — Results table

print(f"{'Approach':<52} {'Score':>8}")
print(f"{'-'*52} {'-'*8}")
for approach, score in [(APPROACH_A, score_a), (APPROACH_B, score_b)]:
    score_str = f"{score:.4f}" if score is not None else "   N/A"
    print(f"{approach['description']:<52} {score_str:>8}")

In [ ]:
# Cell 7 — Plot comparison
import pathlib
import matplotlib.pyplot as plt

RESULTS_DIR = pathlib.Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

labels  = [APPROACH_A["description"], APPROACH_B["description"]]
values  = [score_a if score_a is not None else 0.0,
           score_b if score_b is not None else 0.0]
colours = ["#7bc8f6", "#50fa7b"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, values, color=colours, width=0.45)

ax.set_ylabel("Overall Score", fontsize=12, color="#e8e8e8")
ax.set_ylim(0, 1.1)
ax.set_title(
    "Baseline vs. Novel Contribution — Same Model, Same Framework",
    fontsize=13, pad=16, color="#f5a623",
)
ax.set_facecolor("#0f0f1a")
fig.patch.set_facecolor("#0f0f1a")
ax.tick_params(colors="#e8e8e8", labelsize=9)
ax.spines[:].set_color("#444")

for bar, score in zip(bars, [score_a, score_b]):
    label = f"{score:.3f}" if score is not None else "N/A"
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.03,
        label, ha="center", fontsize=13, fontweight="bold", color="#e8e8e8",
    )

plt.tight_layout()
plt.savefig(RESULTS_DIR / "comparison_plot.png", dpi=150, facecolor=fig.get_facecolor())
plt.show()
print(f"Plot saved → {RESULTS_DIR / 'comparison_plot.png'}")

In [ ]:
# Cell 8 — Export structured comparison report
import json

report = {
    "model": {"url": MODEL_URL, "name": MODEL_NAME},
    "approaches": [
        {
            "description":   APPROACH_A["description"],
            "benchmark":     APPROACH_A["benchmark"],
            "provider":      APPROACH_A["provider"],
            "overall_score": score_a,
            "job_id":        job_id_a,
        },
        {
            "description":   APPROACH_B["description"],
            "benchmark":     APPROACH_B["benchmark"],
            "provider":      APPROACH_B["provider"],
            "overall_score": score_b,
            "job_id":        job_id_b,
        },
    ],
}

report_path = RESULTS_DIR / "comparison_report.json"
report_path.write_text(json.dumps(report, indent=2))
print(f"Report saved → {report_path}")
print()
print("Act 3 complete. Artifacts in results/:")
print("  comparison_plot.png    — figure for paper / presentation")
print("  comparison_report.json — structured data for supplementary materials")